# Кандидатогенерация для поиска услуг Авито — финальное решение

Для каждого запроса бенчмарка решение отбирает из 189 212 объявлений до 50 кандидатов (метрика — Recall@50).

**Схема:** широкий пул кандидатов (BM25 + гео + история + семантический поиск дообученным эмбеддером)
→ ранкер LightGBM LambdaRank → 50 лучших.

Ноутбук выполняется сверху вниз и собирает ответ `answer.csv`. Код — в `src/`, здесь — последовательность шагов
и пояснения. Всё работает локально; сеть нужна только один раз, чтобы скачать открытые веса модели
`intfloat/multilingual-e5-small`.

| Шаг | Время (MacBook M-серии, 16 ГБ) |
|---|---|
| 1. Дообучение эмбеддера (если ещё не обучен) | ~1 ч (MPS) |
| 2–3. Разбиение и эмбеддинги корпуса (если нет кэша) | ~15–25 мин |
| 4. Пул, признаки и обучение ранкера | ~5 мин |
| 5. Ответ на бенчмарк | ~2 мин |

**Результаты по попыткам на платформе:**

| # | Решение | Платформа |
|---|---|---|
| 1 | BM25 + гео | 0.802053 |
| 2 | + пул кандидатов и ранкер LightGBM | 0.852277 |
| 3 | + семантический поиск (e5-small без дообучения) | 0.874677 |
| 4 | + эмбеддер, дообученный на парах «запрос → выбранное объявление» | этот ноутбук |

На локальной валидации дообучение эмбеддера дало к посылке №3 +0.0099 [0.0057; 0.0146] и +0.0124 [0.0086; 0.0170]
на двух независимых разбиениях (парный бутстреп, 95%).

## 0. Окружение и данные

In [1]:
import os
os.environ.setdefault('HF_HUB_OFFLINE', '1')  # после первого скачивания весов — строго офлайн

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from src.common import DATA_DIR, add_query_keys, read_parquet
from src.validation import make_split
from src.dense import EmbeddingCache, DenseIndex
from src import features as F
from src import ranker as R
from src.make_submission import K, check_answer

pd.set_option('display.width', 200)
ROOT = Path.cwd()
OUT_DIR = ROOT / 'submissions' / 'solution'
ENCODER = ROOT / 'artifacts' / 'finetune' / 'e5s_final'
t_start = time.time()

# Данные задания должны лежать в dataset/ (train.parquet, benchmark_queries.parquet, benchmark_items.parquet).
# Описания объявлений train для обучения ранкера не нужны — читаем без них (экономия памяти).
train_cols = [c for c in pq.read_schema(DATA_DIR / 'train.parquet').names if c != 'item_description_raw']
train = add_query_keys(read_parquet(DATA_DIR / 'train.parquet', columns=train_cols))
items = read_parquet(DATA_DIR / 'benchmark_items.parquet')
queries = add_query_keys(read_parquet(DATA_DIR / 'benchmark_queries.parquet'))
print(f'train: {len(train):,} пар запрос–объявление; корпус: {len(items):,}; запросов бенчмарка: {len(queries):,}')

train: 497,673 пар запрос–объявление; корпус: 189,212; запросов бенчмарка: 2,452


## 1. Обучающие запросы ранкера и защита от утечек

Ранкер учится на запросах, отложенных из train тем же протоколом, каким устроен бенчмарк:
- единица запроса — контекст (текст, локация, категория, фильтры, доставка), один контекст на текст;
- 37.5% «знакомых» текстов (другие контексты текста остаются в истории) и 62.5% «новых» (из истории удаляются все
  контексты текста), как в бенчмарке.

**Всё историческое** (гео-переходы, популярность, подкатегории, признаки, пары для дообучения эмбеддера) для этих
запросов считается по истории **без их целевых пар** (`tsplit.history`). При ответе на бенчмарк история — весь train.

In [2]:
tsplit = make_split(train, items, n_queries=R.N_TRAIN_QUERIES, seen_share=0.375, seed=R.SEED + 1, prefix='trn')
h, tq = tsplit.history, tsplit.queries
assert not tq['ctx'].isin(set(h['ctx'])).any(), 'контекст обучающего запроса в истории'
assert not tq.loc[tq['stratum'] == 'new_text', 'q_norm'].isin(set(h['q_norm'])).any(), 'новый текст в истории'
print(f'обучающих запросов: {len(tq):,}; история для их признаков: {len(h):,} строк (из {len(train):,})')

обучающих запросов: 15,000; история для их признаков: 432,509 строк (из 497,673)


## 2. Дообучение эмбеддера

Базовая модель — `intfloat/multilingual-e5-small` (ревизия зафиксирована в `src/dense.py`). Дообучаем её на 60 000
парах «текст запроса → выбранное объявление» из `tsplit.history` — то есть без целей обучающих запросов ранкера:
иначе косинус в их признаках «видел» бы ответы.

Потери — MNRL (остальные объявления батча — негативы; внутри батча не повторяются ни тексты, ни объявления),
1 эпоха, батч 64, lr 2e-5 с разогревом, матрица словаря заморожена (память). Параметры зафиксированы до эксперимента
и не подбирались. Если модель уже обучена (`artifacts/finetune/e5s_final`), шаг пропускается.

In [3]:
if not (ENCODER / 'train_log.json').exists():
    from src.finetune_experiment import stage_train
    stage_train(R.SEED, final=True)  # ~1 ч на MPS; сохраняет модель в artifacts/finetune/e5s_final
log = json.loads((ENCODER / 'train_log.json').read_text())
print(f"пар: {log['n_pairs']:,}, шагов: {log['batches']}, loss: {np.mean(log['losses'][:50]):.3f} → "
      f"{np.mean(log['losses'][-50:]):.3f}; базовая модель {log['base_model']}@{log['base_revision'][:8]}")

пар: 60,000, шагов: 937, loss: 2.004 → 0.499; базовая модель intfloat/multilingual-e5-small@614241f6


## 3. Эмбеддинги корпуса и запросов

Объявление — `passage: <заголовок>. <первые 400 символов описания>`, запрос — `query: <текст>`, до 128 токенов,
векторы нормированы. Эмбеддинги кэшируются в `artifacts/embeddings/` и при повторном запуске не пересчитываются.

In [4]:
emb = EmbeddingCache(str(ENCODER), cache_key=f'finetuned_{ENCODER.name}')
dense_train = DenseIndex(emb.items(tsplit.corpus), emb.queries(tq['search_query']))
dense_bench = DenseIndex(emb.items(items), emb.queries(queries['search_query']))
print(f'эмбеддингов в кэше: объявлений {len(emb.item_row):,}, запросов {len(emb.query_emb):,}')

эмбеддингов в кэше: объявлений 203,789, запросов 17,202


## 4. Пул кандидатов, признаки и ранкер

**Пул** (`FeatureBuilder.channels`), в среднем ~860 объявлений на запрос:

| канал | что берёт |
|---|---|
| local | BM25 (заголовок×2 + параметры + описание, стемминг Snowball), топ-700 в той же локации |
| learned | BM25, топ-300 в 10 локациях, куда по истории чаще всего «уходят» выборы из локации поиска |
| global | BM25, топ-200 по всему корпусу |
| dense_geo / dense_global | семантика, топ-300 в тех же 10 локациях / топ-100 по корпусу |
| history | объявления, которые выбирали по тому же тексту |
| hybrid | baseline: BM25 со своей локацией (для «региональных» локаций без своих объявлений — выученные) |

**Признаки (39):** BM25 по трём индексам и доля слов запроса в объявлении; косинус эмбеддингов; позиции в каналах;
гео (та же локация, расстояние, частота перехода между локациями); подкатегория объявления по словам и тексту
запроса; популярность; выполнение фильтров запроса; рейтинг, отзывы, цена.

**Ранкер:** LightGBM LambdaRank; на запрос — все положительные в пуле, 150 лучших по BM25, 150 случайных и лучшие
по семантике негативы; ранняя остановка по NDCG@50 на 10% обучающих запросов.

In [5]:
tok = F.TokenCache()
fb_train = F.FeatureBuilder(tsplit.corpus, tsplit.history, tok, dense=dense_train)
ds_train = F.build_dataset(fb_train, tq, tsplit.positives, neg_top=R.NEG_TOP, neg_random=R.NEG_RANDOM,
                           seed=R.SEED, drop_no_positive=True, extra_top=R.DENSE_EXTRA_TOP)
model, info = R.train_ranker(ds_train, seed=R.SEED)
print(f"обучающих запросов с положительным в пуле: {len(ds_train.query_ids):,}; строк: {len(ds_train.y):,}; "
      f"деревьев: {info['best_iteration']}")
imp = pd.Series(model.feature_importance('gain'), index=ds_train.feature_names)
display((imp / imp.sum()).sort_values(ascending=False).head(10).round(3).to_frame('доля gain'))
del fb_train, ds_train

обучающих запросов с положительным в пуле: 14,616; строк: 5,056,204; деревьев: 319


,доля gain
rank_hybrid,0.494
loc_share,0.105
dist_km,0.080
rank_dense_geo,0.051
dense_rel,0.044
rank_local,0.038
cov_full,0.020
filter_share,0.019
loc_rank,0.017
mc_p_tok,0.014


## 5. Ответ на бенчмарк: история — весь train, корпус — benchmark_items

In [6]:
fb = F.FeatureBuilder(items, train, tok, dense=dense_bench)
ds = F.build_dataset(fb, queries, positives={})
preds = F.top_by_score(fb, ds, R.predict(model, ds), k=K)
answer = pd.DataFrame({'query_id': queries['query_id'],
                       'answer': [' '.join(preds.get(q, [])) for q in queries['query_id']]})
check_answer(answer, queries, items)  # все требования к формату из условия задачи
OUT_DIR.mkdir(parents=True, exist_ok=True)
answer.to_csv(OUT_DIR / 'answer.csv', index=False)

n_cand = answer['answer'].str.split(' ').str.len()
print(f'запросов: {len(answer):,}; кандидатов на запрос: min {n_cand.min()}, медиана {n_cand.median():.0f}; '
      f'средний размер пула: {np.mean([len(v) for v in ds.pools.values()]):.0f}')
print(f'сохранено: {OUT_DIR / "answer.csv"}; всего {(time.time() - t_start) / 60:.1f} мин')

запросов: 2,452; кандидатов на запрос: min 50, медиана 50; средний размер пула: 863
сохранено: /Users/yaroslavlikh/Desktop/avito-test/avito-test/submissions/solution/answer.csv; всего 118.2 мин


In [7]:
# Воспроизводимость: ответ совпадает с отправленным файлом посылки №4.
sent = ROOT / 'submissions' / '04_ranker_dense_ft' / 'answer.csv'
if sent.exists():
    print('совпадает с submissions/04_ranker_dense_ft/answer.csv байт в байт:',
          (OUT_DIR / 'answer.csv').read_bytes() == sent.read_bytes())

совпадает с submissions/04_ranker_dense_ft/answer.csv байт в байт: True


## Итоги и ограничения

- **Валидация** (5 000 отложенных запросов, по протоколу бенчмарка; решения — по парной разнице вариантов с бутстрепом,
  ключевые — на двух независимых разбиениях): BM25 + гео 0.831 → ранкер 0.892 → семантика 0.922 → дообученный
  эмбеддер 0.932 (seed 42). На платформе значения стабильно ниже локальных на 0.03–0.05, приросты переносились
  частично (+0.061 → +0.050, +0.030 → +0.022).
- **Что не дало устойчивого прироста** и в решение не вошло: признаки «нелокальности» услуги, удвоение семантического
  канала, канал внутри предсказанных подкатегорий, ансамбль из трёх сидов, вдвое больше обучающих запросов, ранняя
  остановка по Recall@50, подкатегории по семантически похожим запросам истории.
- **Ограничения:** размеры каналов и параметры не подбирались; ~3% положительных остаются вне пула; дообучение —
  одна эпоха на 60k пар, без подбора; бенчмарк собран по неизвестному протоколу, поэтому локальная валидация —
  приближение.